In [ ]:
import numpy as np
import re
re = re.compile(rb'[0-9.]+')
a = np.loadtxt(
    '10/m.csv',
    [('df','3B'),('f','3H'),('dx','3B'),('x','3H'),('db','3B'),('dt','3B'),('s','3B'),('t','3B'),('b','3B'),('_','f')],
    delimiter=',',
    converters=lambda x:next(re.finditer(x)).group(),
    skiprows=1,
    usecols=(4,5,0,6,7,8,9,10,0,11,12,13,14,15,0,17,18,0,19,20,21,27,28,29,30,31,32,38)
).view(np.recarray)
a.db[:,2] = 3 - a.db[:,:2].sum(1)
a.dt[:,2] = 3 - a.dt[:,:2].sum(1)
a.df[:,2] = 3 - a.df[:,:2].sum(1)
a.dx[:,2] = 3 - a.dx[:,:2].sum(1)
a.t[:,0] >>= 5
print(a[(t:=a._.argpartition(11)[:10])[a._[t].argsort()]])

t = a[(a.b[:,:2]==1).all(1)]
t.db[:,:2] = t.db[:,1::-1]
a = np.concatenate((a,t)).view(np.recarray)

t = a[(a.b==1).all(1)]
T = t.copy()
t.db[:,::2] = t.db[:,::-2]
T.db[:,1:] = T.db[:,:0:-1]
a = np.concatenate((a,t,T)).view(np.recarray)

t = a[(a.t[:,:2]==1).all(1)]
t.dt[:,:2] = t.dt[:,1::-1]
a = np.concatenate((a,t)).view(np.recarray)

t = a[(a.t==1).all(1)]
T = t.copy()
t.dt[:,::2] = t.dt[:,::-2]
T.dt[:,1:] = T.dt[:,:0:-1]
a = np.concatenate((a,t,T)).view(np.recarray)

t = a[(a.f[:,:2]==1).all(1)]
t.df[:,:2] = t.df[:,1::-1]
a = np.concatenate((a,t)).view(np.recarray)

t = a[(a.f==1).all(1)]
T = t.copy()
t.df[:,::2] = t.df[:,::-2]
T.df[:,1:] = T.df[:,:0:-1]
a = np.concatenate((a,t,T)).view(np.recarray)

t = a[(a.x[:,2:]==1).all(1)]
t.dx[:,1:] = t.dx[:,:0:-1]
a = np.concatenate((a,t)).view(np.recarray)

t = a[(a.x==1).all(1)]
T = t.copy()
t.dx[:,::2] = t.dx[:,::-2]
T.dx[:,:2] = T.dx[:,1::-1]
a = np.concatenate((a,t,T)).view(np.recarray)

a.db_ = a.db[:,0] * 2 + a.db[:,1] - (a.db[:,1] > a.db[:,0])
a.dt_ = a.dt[:,0] * 2 + a.dt[:,1] - (a.dt[:,1] > a.dt[:,0])
a.df_ = a.df[:,0] * 2 + a.df[:,1] - (a.df[:,1] > a.df[:,0])
a.dx_ = a.dx[:,0] * 2 + a.dx[:,1] - (a.dx[:,1] > a.dx[:,0])
a.dfx = a.df_ * 6 + a.dx_
a.b_ = a.b.prod(1)
a.t_ = a.t.prod(1)
a.f_ = a.f.prod(1)
a.x_ = a.x.prod(1)
a = np.rec.fromarrays((a,*a.__dict__.values())).view(
    [(i,j[0])for i,j in a.dtype.fields.items()]+[(i,j.dtype.fields['_'][0] if j.dtype.fields else j.dtype)for i,j in a.__dict__.items()]
)

In [ ]:
import matplotlib.pyplot as plt
for i in a.dtype.fields:
    if i in ('_', 's') or any(i.endswith(j) for j in ('b', 't', 'b_', 't_')):
        continue
    t = a[i]
    if t.ndim == 1:
        t = t[:,None]
    print(i)
    plt.figure(figsize=(t.shape[1] * 2, 2))
    plt.subplots_adjust(0, 0, 1, 1, 0, 0)
    for j, J in enumerate(t.T):
        plt.subplot(1, t.shape[1], j + 1).scatter(J, a._)
        plt.yscale('log')
        if not i.startswith('d'):
            plt.xscale('log')
    plt.show()